In [ ]:
import os
import torch
from datasets import load_dataset
from PIL import Image
from torch.utils.data import Dataset
from transformers import AutoProcessor, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

os.environ["HUGGINGFACE_HUB_TOKEN"] = "<TOKEN>"
HF_TOKEN = os.getenv("HUGGINGFACE_HUB_TOKEN")

ROOT = "DatasetFinal"
TRAIN_JSONL = "DatasetFinal/splits/stage2_llm/train.jsonl"
VAL_JSONL = "DatasetFinal/splits/stage2_llm/val.jsonl"
MODEL_NAME = "google/gemma-3-4b-it"
STAGE1_ADAPTER = "stage1_vision_lora_adapter"
CHECKPOINT_DIR = "stage2_llm_checkpoints"
FINAL_ADAPTER_DIR = "stage2_llm_lora_adapter"



In [ ]:
train_ds = load_dataset("json", data_files=TRAIN_JSONL)["train"]
val_ds = load_dataset("json", data_files=VAL_JSONL)["train"]



In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_NAME, token=HF_TOKEN, trust_remote_code=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
)



In [ ]:
model = PeftModel.from_pretrained(base_model, STAGE1_ADAPTER)
model = prepare_model_for_kbit_training(model)

for name, param in model.named_parameters():
    if ("lora" in name.lower() and "vision" in name.lower()) or "vision_tower" in name or "multi_modal_projector" in name or "mm_projector" in name:
        param.requires_grad = False



In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=r".*language_model.*(q_proj|k_proj|v_proj|o_proj)",
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.config.use_cache = False



In [ ]:
class ChessStrategyDataset(Dataset):
    def __init__(self, hf_dataset, root_folder, processor):
        self.ds = hf_dataset
        self.root = root_folder
        self.processor = processor
        self.tokenizer = processor.tokenizer
        self.image_token_id = self.tokenizer.convert_tokens_to_ids("<image>") or 262144

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        user_msg = item["messages"][0]["content"]
        assistant_msg = item["messages"][1]["content"]
        
        image = Image.open(os.path.join(self.root, user_msg[0]["image"])).convert("RGB")
        
        messages = [
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_msg[1]["text"].strip()}]},
            {"role": "assistant", "content": [{"type": "text", "text": assistant_msg[0]["text"].strip()}]},
        ]
        
        chat_text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        enc = self.processor(text=chat_text, images=image, return_tensors="pt")
        
        input_ids = enc["input_ids"].squeeze(0)
        labels = input_ids.clone()
        
        assistant_start = chat_text.rfind("<assistant>")
        if assistant_start != -1:
            prefix_ids = self.tokenizer(chat_text[:assistant_start], add_special_tokens=False)["input_ids"]
            labels[:len(prefix_ids)] = -100
        
        labels[labels == self.image_token_id] = -100
        
        return {
            "input_ids": input_ids,
            "attention_mask": enc["attention_mask"].squeeze(0),
            "pixel_values": enc["pixel_values"].squeeze(0),
            "labels": labels
        }



In [ ]:
def collate_fn(batch):
    padded = processor.tokenizer.pad(
        {"input_ids": [b["input_ids"] for b in batch], "attention_mask": [b["attention_mask"] for b in batch]},
        padding=True,
        return_tensors="pt",
    )
    
    labels_padded = torch.full((len(batch), padded["input_ids"].shape[1]), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        labels_padded[i, :b["labels"].shape[0]] = b["labels"]
    
    return {
        "input_ids": padded["input_ids"],
        "attention_mask": padded["attention_mask"],
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        "labels": labels_padded,
    }

train_dataset = ChessStrategyDataset(train_ds, ROOT, processor)
val_dataset = ChessStrategyDataset(val_ds, ROOT, processor)



In [ ]:
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    weight_decay=0.01,
    fp16=True,
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=3,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to="tensorboard",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)



In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=processor.tokenizer,
    data_collator=collate_fn,
)

trainer.train()



In [ ]:
model.save_pretrained(FINAL_ADAPTER_DIR)
processor.save_pretrained(FINAL_ADAPTER_DIR)